In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from IPython.display import Audio, display

2026-04-19 20:33:18.033403: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-19 20:33:18.050490: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-19 20:33:18.050512: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-19 20:33:18.051047: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-19 20:33:18.054307: I tensorflow/core/platform/cpu_feature_guar

In [11]:
# Se carga YAMNet para generar embedding, el mejor clasificador entrenado y el mapa de las clases.

YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"

yamnet_model = hub.load(YAMNET_HANDLE)
classifier = tf.keras.models.load_model("models/best_temporal_classifier.keras")

class_mapping = pd.read_csv("models/class_mapping.csv")
id_to_name = dict(zip(class_mapping["class_id"], class_mapping["label"]))

print("YAMNet cargado")
print("Clasificador cargado")

YAMNet cargado
Clasificador cargado


In [12]:
# Se definen funciones auxiliares.
# El modelo funciona con audios de 6 segundos, si son más largos se cortan y si son más cortos se paddean 

TARGET_SR = 16000
TARGET_SECONDS = 6
TARGET_LEN = TARGET_SR * TARGET_SECONDS


def load_audio_16k_mono(filepath):
    wav, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
    wav = wav.astype(np.float32)
    return wav


def make_fixed_6s_clip(wav, target_len=TARGET_LEN, seed=None):
    """
    Devuelve un clip fijo de 6 s.

    - Si el audio dura >= 6 s: toma una ventana aleatoria de 6 s.
    - Si dura < 6 s: repite el audio y toma una ventana de 6 s
      empezando en una posición aleatoria dentro del propio patrón repetido.
    """
    if len(wav) == 0:
        raise ValueError("El audio está vacío.")

    rng = np.random.default_rng(seed)

    if len(wav) >= target_len:
        max_start = len(wav) - target_len
        start = rng.integers(0, max_start + 1)
        clip = wav[start:start + target_len]
        return clip

    repeats = int(np.ceil((target_len + len(wav)) / len(wav)))
    tiled = np.tile(wav, repeats)

    start = rng.integers(0, len(wav))
    clip = tiled[start:start + target_len]

    if len(clip) < target_len:
        clip = np.pad(clip, (0, target_len - len(clip)))

    return clip.astype(np.float32)

In [ ]:
# El clasificador recibe los embedding concatenados.

def build_concatenated_embedding_from_wav(wav, yamnet_model, classifier_clip):
    """
    Convierte un wav de 6 s en el vector concatenado que espera el clasificador.
    Ajusta por padding/truncado si hiciera falta.
    """
    scores, embeddings, spectrogram = yamnet_model(wav)   # [T, 1024]
    embeddings_np = embeddings.numpy()

    emb_dim = embeddings_np.shape[1]   # normalmente 1024
    input_dim = classifier_clip.input_shape[-1]
    expected_frames = input_dim // emb_dim

    current_frames = embeddings_np.shape[0]

    if current_frames < expected_frames:
        pad = np.zeros((expected_frames - current_frames, emb_dim), dtype=embeddings_np.dtype)
        embeddings_np = np.vstack([embeddings_np, pad])
    elif current_frames > expected_frames:
        embeddings_np = embeddings_np[:expected_frames]

    concat_embedding = embeddings_np.reshape(1, expected_frames * emb_dim)

    return concat_embedding, expected_frames, emb_dim

In [9]:
# Función de inferencia por clip de audio

def predict_audio_file_concat(filepath, yamnet_model, classifier_clip, id_to_name=None, seed=42):
    wav = load_audio_16k_mono(filepath)
    clip = make_fixed_6s_clip(wav, target_len=TARGET_LEN, seed=seed)

    concat_embedding, n_frames, emb_dim = build_concatenated_embedding_from_wav(
        wav=clip,
        yamnet_model=yamnet_model,
        classifier_clip=classifier_clip
    )

    logits = classifier_clip.predict(concat_embedding, verbose=0)[0]
    probs = tf.nn.softmax(logits).numpy()

    pred_class_id = int(np.argmax(probs))
    pred_class_name = id_to_name.get(pred_class_id, str(pred_class_id)) if id_to_name else str(pred_class_id)

    probs_df = pd.DataFrame({
        "class_id": np.arange(len(probs)),
        "probability": probs
    }).sort_values("probability", ascending=False).reset_index(drop=True)

    if id_to_name:
        probs_df["class_name"] = probs_df["class_id"].map(id_to_name)

    return {
        "filepath": filepath,
        "wav_original": wav,
        "wav_6s": clip,
        "pred_class_id": pred_class_id,
        "pred_class_name": pred_class_name,
        "probabilities": probs_df,
        "n_frames": n_frames,
        "emb_dim": emb_dim,
        "concat_shape": concat_embedding.shape,
    }

In [17]:
# Se utiliza con un audio.


audio_path ="test_data/PAJARO_en_Paridae_Saxicola_rubetra_France_2010-06-23_XC648317_song__segundo_8.4.wav"                   #"/ruta/a/tu/audio.wav"

result = predict_audio_file_concat(
    filepath=audio_path,
    yamnet_model=yamnet_model,
    classifier_clip=classifier,
    id_to_name=id_to_name,
    seed=42
)

print("Archivo:", result["filepath"])
print("Frames YAMNet:", result["n_frames"])
print("Clase predicha:", result["pred_class_id"], "-", result["pred_class_name"])
display(result["probabilities"])

Archivo: test_data/PAJARO_en_Paridae_Saxicola_rubetra_France_2010-06-23_XC648317_song__segundo_8.4.wav
Frames YAMNet: 12
Clase predicha: 2 - Paridae_Saxicola_rubetra


,class_id,probability,class_name
0,2,0.715886,Paridae_Saxicola_rubetra
1,5,0.232885,Troglodytidae_Troglodytes_aedon
2,3,0.017222,Paridae_Saxicola_rubicola
3,9,0.015367,Turdidae_Catharus_aurantiirostris
4,0,0.010111,Fringillidae_Serinus_serinus
5,1,0.002993,Paridae_Hypocnemis_peruviana
6,4,0.002543,Sin_Pajaro
7,13,0.001212,Turdidae_Catharus_ustulatus
8,7,0.000673,Troglodytidae_Troglodytes_pacificus
9,8,0.000536,Troglodytidae_Troglodytes_troglodytes


In [19]:
audio_path ="test_data/PAJARO_en_Troglodytidae_Troglodytes_aedon_Brazil_1999-05-08_XC303926_song__segundo_0.0.wav"                   #"/ruta/a/tu/audio.wav"

result = predict_audio_file_concat(
    filepath=audio_path,
    yamnet_model=yamnet_model,
    classifier_clip=classifier,
    id_to_name=id_to_name,
    seed=42
)

print("Archivo:", result["filepath"])
print("Frames YAMNet:", result["n_frames"])
print("Clase predicha:", result["pred_class_id"], "-", result["pred_class_name"])
display(result["probabilities"])

Archivo: test_data/PAJARO_en_Troglodytidae_Troglodytes_aedon_Brazil_1999-05-08_XC303926_song__segundo_0.0.wav
Frames YAMNet: 12
Clase predicha: 2 - Paridae_Saxicola_rubetra


,class_id,probability,class_name
0,2,0.508727,Paridae_Saxicola_rubetra
1,5,0.470922,Troglodytidae_Troglodytes_aedon
2,3,0.008351,Paridae_Saxicola_rubicola
3,13,0.007524,Turdidae_Catharus_ustulatus
4,11,0.003619,Turdidae_Catharus_fuscescens
5,1,0.000287,Paridae_Hypocnemis_peruviana
6,9,0.000273,Turdidae_Catharus_aurantiirostris
7,0,0.000081,Fringillidae_Serinus_serinus
8,8,0.000051,Troglodytidae_Troglodytes_troglodytes
9,4,0.000046,Sin_Pajaro
